# Energy demand forecasting: feature engineering & EDA


## Setup & configuration

In [ ]:
import warnings
from dataclasses import dataclass, field
from typing import Literal, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
CSV_PATH = "half_hourly.csv"

# Configuration object

In [ ]:
@dataclass
class FeatureConfig:
    timestamp_col: str
    target_col: str

    # try to use interval-START so that the first period is 00:00-00:30 everywhere.
    timestamp_label: Literal["start", "end"] = "start"

    tz: Optional[str] = None

    freq: Optional[str] = None

    lags: tuple[str, ...] = ("30min", "1h", "1D", "7D")
    rolling_windows: tuple[str, ...] = ("3h", "1D", "7D")

    country: Optional[str] = None

    min_valid_value: float = 0.0

    cyclical: tuple[str, ...] = ("time_of_day", "day_of_week", "day_of_year")

    extra_drop: tuple[str, ...] = field(default_factory=tuple)

### Transformation: raw frame to regular, interval-start indexed series

- Parse timestamps 
- Handle timezone 
- Force target to numeric and flag non-positive values 
- Sort & remove duplicate 
- Infer sampling interval 
- Transform interval-end stamps to interval-start for parity 
- Form a complete matrix with gaps in lagging and rolling features becoming NaN rows, flagged as `is_imputed_row`.

In [ ]:
def read_freq(idx: pd.DatetimeIndex) -> pd.Timedelta:
    
    step = pd.Series(idx).diff().dropna()
    if step.empty:
        raise ValueError("Not enough timestamps to infer frequency.")
    return step.mode().iloc[0]


def read_label(idx: pd.DatetimeIndex, step: pd.Timedelta) -> Optional[str]:
    
    first, last = idx[0], idx[-1]
    first_off = first - first.normalize()
    last_off = last - last.normalize()
    if first_off == step and last_off == pd.Timedelta(0):
        return "end"
    if first_off == pd.Timedelta(0):
        return "start"
    return None

def clean_series(df: pd.DataFrame, cfg: FeatureConfig) -> tuple[pd.DataFrame, dict]:
    """
    Returns a cleaned frame with interval-start timestamps and 
    'is_imputed_row' column to mark rows with NaN data
    Also returns a report describing the data
    """
    for c in (cfg.timestamp_col, cfg.target_col):
        if c not in df.columns:
            raise KeyError(f"Column '{c}' not found. Available: {list(df.columns)}")

    report: dict = {"rows_in": len(df)}
    work = df[[cfg.timestamp_col, cfg.target_col]].copy()

    # timestamps
    ts = pd.to_datetime(work[cfg.timestamp_col], errors="coerce", utc=False)
    
    # normalise via UTC.
    if ts.dtype == object:
        ts = pd.to_datetime(work[cfg.timestamp_col], errors="coerce", utc=True)
    report["unparseable_timestamps"] = int(ts.isna().sum())

    if ts.dt.tz is None:
        if cfg.tz:
            ts = ts.dt.tz_localize(cfg.tz, ambiguous="NaT", nonexistent="NaT")
    elif cfg.tz:
        ts = ts.dt.tz_convert(cfg.tz)

    work.index = pd.DatetimeIndex(ts, name="timestamp")
    work = work.drop(columns=[cfg.timestamp_col])
    work = work[work.index.notna()]

    # target column 
    work[cfg.target_col] = pd.to_numeric(work[cfg.target_col], errors="coerce")
    report["non_numeric_or_null"] = int(work[cfg.target_col].isna().sum())
    bad = work[cfg.target_col] <= cfg.min_valid_value
    report["at_or_below_min_valid"] = int(bad.sum())
    work.loc[bad, cfg.target_col] = np.nan

    # order & remove duplicates 
    if not work.index.is_monotonic_increasing:
        report["was_unsorted"] = True
        work = work.sort_index()
    dups = work.index.duplicated(keep="last")
    report["duplicate_timestamps"] = int(dups.sum())
    work = work[~dups]

    # frequency & interval label 
    step = pd.Timedelta(cfg.freq) if cfg.freq else read_freq(work.index)
    report["freq"] = str(step)

    label = read_label(work.index, step)
    report["label"] = label
    if label and label != cfg.timestamp_label:
        warnings.warn(
            f"Config timestamp_label='{cfg.timestamp_label}'and data dont match")

    if cfg.timestamp_label == "end":
        work.index = work.index - step
    work.index.name = "timestamp"

    # complete grid 
    full = pd.date_range(work.index.min(), work.index.max(), freq=step)
    n_before = len(work)
    work = work.reindex(full)
    work.index.name = "timestamp"
    report["gap_rows_inserted"] = int(len(full) - n_before)
    work["is_imputed_row"] = work[cfg.target_col].isna()
    report["missing_target_rows"] = int(work["is_imputed_row"].sum())
    report["rows_out"] = len(work)
    report["start"], report["end"] = str(work.index[0]), str(work.index[-1])
    report["span_days"] = round((work.index[-1] - work.index[0] + step) / pd.Timedelta("1D"), 2)
    return work, report

### Calendar features

In [ ]:
def add_calendar_features(df: pd.DataFrame, cfg: FeatureConfig) -> pd.DataFrame:
    out = df.copy()
    idx = out.index
    out["time_of_day"] = idx.hour + idx.minute / 60.0
    out["hour"] = idx.hour
    out["day_of_week"] = idx.dayofweek                       # Monday=0
    out["is_weekend"] = (idx.dayofweek >= 5).astype(int)
    out["day_of_month"] = idx.day
    out["day_of_year"] = idx.dayofyear
    out["week_of_year"] = idx.isocalendar().week.astype(int).values
    out["month"] = idx.month
    out["quarter"] = idx.quarter
    out["year"] = idx.year

    # cyclical features for distance based regression algorithms
    period = {"time_of_day": 24.0, "day_of_week": 7.0, "day_of_year": 365.25,
              "month": 12.0}
    for col in cfg.cyclical:
        p = period[col]
        out[f"{col}_sin"] = np.sin(2 * np.pi * out[col] / p)
        out[f"{col}_cos"] = np.cos(2 * np.pi * out[col] / p)
    
    return out